# X25 Stage 1 measurement — in-place TTT vs zero-update control

20 frozen Gutenberg stories x 2 arms. Inference-only GPU minutes. Writes `x25_results.json`. No thresholds in Stage 1 — this measures the baseline variance the amendment will set bars from.

In [ ]:
import glob, os, shutil, subprocess, sys
work = '/kaggle/working/kalia'
if os.path.exists(work):
    shutil.rmtree(work)
r = subprocess.run(['git', 'clone', '--quiet', '--branch', 'main', 'https://github.com/subhajitlucky/kalia.git', work], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
os.chdir(work)
print('code ready')
ckpt_hits = sorted(glob.glob('/kaggle/input/**/ckpt.pt', recursive=True))
assert ckpt_hits, 'attach kalia-v020-ckpt'
CKPT = ckpt_hits[0]
print('checkpoint:', CKPT)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import json
import tiktoken
import torch
from eval_entities import entity_report
from sample import load_model
from ttt import generate_with_ttt
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
enc = tiktoken.get_encoding('gpt2')
STORIES = json.loads(open('eval/ttt_probe_stories.json').read())
print('stories:', len(STORIES))
model = load_model(CKPT, DEVICE)
print('checkpoint loaded')

In [ ]:
results = []
for story in STORIES:
    prompt_ids = torch.tensor([enc.encode(story['prompt'])], dtype=torch.long)
    row = {'id': story['id']}
    for tag, lr in (('control', 0.0), ('ttt', 1e-4)):
        out, diag = generate_with_ttt(model, prompt_ids.to(DEVICE), max_new_tokens=1024, chunk_tokens=256, ttt_lr=lr, temperature=0.8, top_k=200, seed=7)
        continuation = enc.decode(out[0, prompt_ids.size(1):].tolist())
        report = entity_report(story['prompt'], continuation)
        row[tag] = {'retention': report['retention'], 'updates': diag['updates']}
    results.append(row)
    print(f"{story['id']}: control={row['control']['retention']:.4f} ttt={row['ttt']['retention']:.4f}", flush=True)
json.dump(results, open('/kaggle/working/x25_results.json', 'w'), indent=2)
print('wrote x25_results.json')

In [ ]:
import statistics
controls = [r['control']['retention'] for r in results]
ttts = [r['ttt']['retention'] for r in results]
deltas = [t - c for t, c in zip(ttts, controls)]
print(f"control mean={statistics.mean(controls):.4f} stdev={statistics.pstdev(controls):.4f}")
print(f"ttt     mean={statistics.mean(ttts):.4f} stdev={statistics.pstdev(ttts):.4f}")
print(f"delta   mean={statistics.mean(deltas):+.4f} stdev={statistics.pstdev(deltas):.4f}")
print('Stage 1 output: baseline variance + measured delta. No threshold applied.')